<a href="https://colab.research.google.com/github/Laxitha/Rennaigan/blob/backend-gateway/rennaigan_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rennaigan backend on Colab

Runs the five detector services and the gateway on a Colab GPU and gives you a public
`https://….trycloudflare.com` address to paste into the UI (**Settings → ML Gateway → Gateway URL**).

Before running: **Runtime → Change runtime type → T4 GPU**. Then **Runtime → Run all**.
The first run takes about 8 minutes (model downloads). Keep this tab open while you use the UI.

Cell 3 prints the address. If something fails, run the last cell and copy its output.

After a code update: re-run cell 3. It pulls the latest code, restarts the backend and prints the address again (it stays the same while this session lives).

In [1]:
#@title 1. Get the code
import os
if not os.path.exists("/content/Rennaigan"):
    !git clone -q --depth 1 -b backend-gateway https://github.com/Laxitha/Rennaigan.git /content/Rennaigan
os.chdir("/content/Rennaigan")
!git pull -q
!git log --oneline -1
!nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null || echo "No GPU: analysis will run in the slower reduced mode. For full speed: Runtime > Change runtime type > T4 GPU."

53f2519 (HEAD -> backend-gateway, origin/backend-gateway) Uploads: a long file name keeps its extension when shortened
Tesla T4


In [4]:
#@title 2. One-time setup: tools, model repositories, packages and weights (about 6 minutes)
import os
os.chdir("/content/Rennaigan")
!bash scripts/colab_setup.sh


=== System tools
exiftool ok
c2patool ok

=== Model repositories
lipforensics  sbi  syncnet  trufor

=== Python packages (image, video, motion, metadata services)
ok      efficientnet_pytorch
ok      transformers
ok      insightface
ok      face_alignment
ok      mediapipe
ok      scenedetect
ok      yacs
ok      anthropic

=== Model weights
^C


In [ ]:
#@title 3. Start the backend and get the public address
import os
os.chdir("/content/Rennaigan")

# Optional: a reasoned real/deepfake verdict with a written review, from Gemini.
# Get a key at https://aistudio.google.com/apikey, then add it under the key icon in Colab's
# left sidebar as a secret named GEMINI_API_KEY and switch on "Notebook access".
# Never paste the key into a cell. (A secret named ANTHROPIC_API_KEY is used if there is no Gemini one.)
from google.colab import userdata
for name in ("GEMINI_API_KEY", "ANTHROPIC_API_KEY"):
    try:
        key = userdata.get(name)
    except Exception:
        key = None  # this secret is not set
    if key:
        os.environ[name] = key.strip()
        # Also send the image (or up to six video frames) for a visual review. This shares
        # uploaded media with the provider; delete the next line to send detector findings only.
        os.environ["RENNAIGAN_ASSESS_SEND_MEDIA"] = "1"

!git pull -q
!python scripts/colab.py up


Reasoned verdict: Gemini key present
gateway v1.0.0: 5/5 services, 7 cases, reasoned assessment: {'enabled': True, 'provider': 'gemini', 'model': 'gemini-3.8-flash', 'send_media': True}
  online  image     http://127.0.0.1:8001
  online  video     http://127.0.0.1:8002
  online  audio     http://127.0.0.1:8003
  online  metadata  http://127.0.0.1:8004
  online  motion    http://127.0.0.1:8005

GATEWAY URL: https://occupation-indoor-positioning-timber.trycloudflare.com 
Paste it into the UI: Settings -> ML Gateway -> Save & Connect.
Loading models (a few minutes on a fresh runtime)...
Models loaded.


In [ ]:
#@title 4. Optional self-test: every detector on a real talking-head clip
import os
os.chdir("/content/Rennaigan")
!python scripts/colab.py selftest

In [ ]:
#@title Status and logs (run when something looks wrong)
import os
os.chdir("/content/Rennaigan")
!python scripts/colab.py status
!tail -15 /content/backend.log